# scVI default - nph_cerebellum

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR

This notebook is intentionally unexecuted.


In [1]:
import gc
import inspect
import json
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
PROJECT_ROOT = (
    "/ubc/cs/research/beaver/projects/"
    "Single-cell-integration-metrics"
)
os.environ["R_HOME"] = (
    f"{PROJECT_ROOT}/CondaEnv/scvi_py310/lib/R"
)
os.environ["R_LIBS_SITE"] = (
    f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R/library"
)
sys.path.insert(0, f"{PROJECT_ROOT}/scib")

OPTIMIZER_PATH = Path(
    PROJECT_ROOT,
    "scripts",
    "optuna_multi_metrics",
    "optuna_scvi",
    "optimize_scvi_multi_metrics.py",
)
sys.path.insert(0, str(OPTIMIZER_PATH.parent))

import anndata as ad
import numpy as np
import pandas as pd
import torch
import optimize_scvi_multi_metrics as scvi_workflow

assert Path(scvi_workflow.__file__).resolve() == OPTIMIZER_PATH


In [2]:
DATASET_NAME = 'nph_cerebellum'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEYS = ('Sample',)
BATCH_KEY = BATCH_KEYS[0]
LABEL_KEY = 'Celltype'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_NEIGHBORS = 15
USE_HVG = True
N_TOP_HVG = 2000
MAX_EPOCHS = 400

EMBED_KEY = "X_scvi"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "scvi_metric_batch"
METRIC_LABEL_KEY = "scvi_metric_label"

assert DATASET_NAME == 'nph_cerebellum'
assert BATCH_KEYS == ('Sample',)
assert LABEL_KEY == 'Celltype'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_keys": list(BATCH_KEYS),
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
        "max_epochs": MAX_EPOCHS,
    },
    name="configuration",
)


dataset                                          nph_cerebellum
data_path     /ubc/cs/research/beaver/projects/Single-cell-i...
batch_keys                                             [Sample]
label_key                                              Celltype
metrics                [BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR]
seeds                            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
use_hvg                                                    True
n_top_hvg                                                  2000
max_epochs                                                  400
Name: configuration, dtype: object

In [3]:
scvi_workflow.DATA_PATH = DATA_PATH
scvi_workflow.DATASET_NAME = DATASET_NAME
scvi_workflow.BATCH_KEYS = list(BATCH_KEYS)
scvi_workflow.BATCH_KEY = BATCH_KEY
scvi_workflow.LABEL_KEY = LABEL_KEY
scvi_workflow.USE_HVG = USE_HVG
scvi_workflow.N_TOP_HVG = N_TOP_HVG
scvi_workflow.MAX_EPOCHS = MAX_EPOCHS
scvi_workflow.N_NEIGHBORS = N_NEIGHBORS
scvi_workflow.EMBED_KEY = EMBED_KEY
scvi_workflow.CLUSTER_KEY = CLUSTER_KEY
scvi_workflow.METRIC_BATCH_KEY = METRIC_BATCH_KEY
scvi_workflow.METRIC_LABEL_KEY = METRIC_LABEL_KEY

scvi_workflow.USE_BAMI = "BAMI" in SELECTED_METRICS
scvi_workflow.USE_CASW = "CASW" in SELECTED_METRICS
scvi_workflow.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
scvi_workflow.USE_TRAJ = "TRAJ" in SELECTED_METRICS
scvi_workflow.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
scvi_workflow.USE_BRAS = "BRAS" in SELECTED_METRICS
scvi_workflow.USE_KBET = "KBET" in SELECTED_METRICS
scvi_workflow.USE_CPCR = "CPCR" in SELECTED_METRICS

scvi_workflow.validate_configuration(SELECTED_METRICS)
scvi_workflow.validate_dataset_contents()

pd.Series(
    {
        "optimizer": str(OPTIMIZER_PATH),
        "scvi_version": scvi_workflow.scvi.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    name="preflight",
)


Dataset preflight: n_obs=84619 n_batches=2 n_labels=8 counts_layer=True
CPCR preflight: eligible_labels=8


optimizer         /ubc/cs/research/beaver/projects/Single-cell-i...
scvi_version                                                  1.3.3
cuda_available                                                 True
Name: preflight, dtype: object

In [4]:
constructor_signature = inspect.signature(
    scvi_workflow.scvi.model.SCVI.__init__
)
training_signature = inspect.signature(
    scvi_workflow.scvi.train.TrainingPlan.__init__
)
assert scvi_workflow.scvi.__version__ == "1.3.3"
assert constructor_signature.parameters["n_latent"].default == 10
assert constructor_signature.parameters["n_hidden"].default == 128
assert constructor_signature.parameters["n_layers"].default == 1
assert constructor_signature.parameters["dropout_rate"].default == 0.1
assert training_signature.parameters["lr"].default == 0.001

SCVI_PARAMS = {'n_latent': 10, 'n_hidden': 128, 'n_layers': 1, 'dropout_rate': 0.1, 'lr': 0.001}
assert tuple(SCVI_PARAMS) == ('n_latent', 'n_hidden', 'n_layers', 'dropout_rate', 'lr')

pd.Series(
    {
        **SCVI_PARAMS,
        "gene_likelihood": "zinb",
        "dispersion": "gene",
        "latent_distribution": "normal",
        "early_stopping": True,
        "max_epochs": MAX_EPOCHS,
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
    },
    name="default_setting",
)

def run_scvi_setting(adata, seed):
    return scvi_workflow.run_scvi(
        adata,
        n_latent=SCVI_PARAMS["n_latent"],
        n_hidden=SCVI_PARAMS["n_hidden"],
        n_layers=SCVI_PARAMS["n_layers"],
        dropout_rate=SCVI_PARAMS["dropout_rate"],
        lr=SCVI_PARAMS["lr"],
        seed=int(seed),
    )


In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    result = scvi_workflow.compute_joint_metrics(adata_raw, adata_int)
    if tuple(result) != (
        *SELECTED_METRICS,
        "bio_score",
        "batch_score",
        "joint_score",
    ):
        raise ValueError(f"Unexpected metric result keys: {tuple(result)}")
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_scvi_setting(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


[rank: 0] Seed set to 0


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.7694550718580795


optimised clustering against scvi_metric_label
optimal cluster resolution: 0.6
optimal score: 0.8126177602005084
BRAS...


seed           9.000000
BAMI           0.812618
CASW           0.624216
SCGRAPHM       0.639705
BRAS           0.824654
KBET           0.302050
CPCR           0.001137
bio_score      0.692180
batch_score    0.375947
joint_score    0.565687
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.825493,0.621871,0.721818,0.828683,0.334861,0.001755,0.723061,0.388433,0.589210
1,1,0.840505,0.627034,0.602255,0.826289,0.315150,0.000257,0.689931,0.380565,0.566185
2,2,0.814143,0.622572,0.691986,0.833112,0.305531,0.001805,0.709567,0.380150,0.577800
3,3,0.838449,0.626155,0.714228,0.844324,0.344732,0.003206,0.726277,0.397421,0.594735
4,4,0.824572,0.625614,0.650177,0.839333,0.361931,0.005679,0.700121,0.402314,0.580998
5,5,0.806656,0.625214,0.679569,0.840681,0.365309,0.002011,0.703813,0.402667,0.583355
6,6,0.817461,0.620137,0.678378,0.839204,0.331359,0.004451,0.705325,0.391671,0.579864
7,7,0.817165,0.624317,0.709250,0.827980,0.324315,0.003152,0.716911,0.385149,0.584206
8,8,0.799384,0.626700,0.708217,0.830368,0.327421,0.002222,0.711434,0.386671,0.581529
9,9,0.812618,0.624216,0.639705,0.824654,0.302050,0.001137,0.692180,0.375947,0.565687


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.819645,0.012317,0.799384,0.840505
CASW,0.624383,0.002131,0.620137,0.627034
SCGRAPHM,0.679558,0.036434,0.602255,0.721818
BRAS,0.833463,0.006551,0.824654,0.844324
KBET,0.331266,0.020328,0.302050,0.365309
CPCR,0.002568,0.001518,0.000257,0.005679
bio_score,0.707862,0.011480,0.689931,0.726277
batch_score,0.389099,0.008845,0.375947,0.402667
joint_score,0.580357,0.008565,0.565687,0.594735
